# Track 9 Project: Clean the Messy Dataset

**Goal:** Build a leak-free Scikit-Learn `ColumnTransformer` + `Pipeline` that takes a messy tabular dataset (with missing values, skewed numeric features, outliers, high-cardinality categories, and class imbalance) and trains a clean classifier.

### Milestones
1. **EDA & Train/Test Split First:** Inspect missingness, skew, and class balance **only after** splitting into train and test.
2. **Leak-Free ColumnTransformer:** Handle numeric medians + missingness indicators, categorical `"Unknown"` + encoding, and cyclical/engineered features inside a `Pipeline`.
3. **Imbalance & Honest Evaluation:** Compare plain accuracy vs PR-AUC / F1 with `class_weight="balanced"` on the untouched test set.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, average_precision_score

# Load the Ames Housing / Kaggle Credit Card Fraud / messy tabular CSV
# Example synthetic messy dataset fallback so this notebook runs immediately:
rng = np.random.default_rng(42)
n = 1000
df = pd.DataFrame({
    "income_k": np.where(rng.random(n) < 0.15, np.nan, rng.lognormal(mean=3.2, sigma=0.6, size=n)),
    "age": np.where(rng.random(n) < 0.08, np.nan, rng.integers(18, 70, size=n)),
    "city": rng.choice(["Bengaluru", "Pune", "Mumbai", "Delhi", None], size=n, p=[0.35, 0.25, 0.2, 0.1, 0.1]),
    "hour": rng.integers(0, 24, size=n),
})
# Target: imbalanced default/approval signal correlated with missing income (MNAR)
prob = np.where(df["income_k"].isna(), 0.35, 0.05)
df["target"] = (rng.random(n) < prob).astype(int)
df.head()

## Milestone 1: Split First, Then Inspect
Never fit scalers, imputers, or encoders before `train_test_split`.

In [ ]:
# Cyclical encoding for hour (deterministic, no leakage)
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24.0)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24.0)

X = df.drop(columns=["target", "hour"])
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print("Train missingness:\n", X_train.isna().mean().round(3))
print("\nClass balance (train):\n", y_train.value_counts(normalize=True).round(3))

## Milestone 2: Build the Leak-Free `ColumnTransformer` + `Pipeline`

In [ ]:
numeric_cols = ["income_k", "age", "hour_sin", "hour_cos"]
categorical_cols = ["city"]

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", RobustScaler()),
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", min_frequency=10)),
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric_cols),
    ("cat", categorical_pipe, categorical_cols),
])

model = Pipeline([
    ("prep", preprocessor),
    ("clf", LogisticRegression(class_weight="balanced", max_iter=1000)),
])

model.fit(X_train, y_train)

## Milestone 3: Honest Evaluation on Untouched Test Set

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, digits=3))
print(f"PR-AUC (Average Precision): {average_precision_score(y_test, y_prob):.4f}")